# Menneske-i-loopen: Forhandling før handling, risikokategorisering og revisionslogning

README-filen til denne lektion introducerer Menneske-i-loopen med et kort uddrag, der beder brugeren godkende (`APPROVE`) eller afvise (`REJECT`), efter agenten allerede har produceret et svar. Dette mønster er et fint udgangspunkt, men produktions-implementeringer af HITL har ofte brug for tre yderligere elementer:

1. En **forhandling før handling**, som kører **før** agenten udfører et risikabelt trin, så omkostninger, irreversibilitet og forsinkelse holdes under kontrol.
2. **Risikokategorisering**, så lavrisikohandlinger udføres automatisk, mellemrisikohandlinger godkendes i batch, og kun højriskohandlinger blokeres for en menneskelig godkendelse.
3. En **revisionslog plus revisionssløjfe**, så hver beslutning ved forhandlingen registreres som JSONL, og en afvisning genstarter agenten med en struktureret årsag i stedet for blot at printe `Revising...`.

Denne notebook bygger hver af disse ovenpå de samme primitive som `06-system-message-framework.ipynb`. Den kører end-to-end i `DEMO_MODE = True` (ingen interaktiv input nødvendig) eller med ægte `input()`-prompt, når `DEMO_MODE = False`. Bemærk: i DEMO_MODE er genafsøgningen fra det tredje mål scriptet, så sløjfemekanikken er synlig end-to-end. Ægte revisionsdrevet rekategorisering kræver `DEMO_MODE = False` og en operatør.

**Uden for scope (håndteres i andre lektioner):** autentificering og adgangskontrol (Lesson 06 README threat 2), middleware til værktøjskald (Lesson 14 MAF deep dive), debatmønstre med flere agenter.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Mønster 1: Forhandling før handling

README'ets HITL-udsnit kalder agenten først og beder derefter brugeren om at godkende outputtet. Det er et **post-handlings** flow. Agenten har allerede udført handlingen, så LLM-kaldets omkostning er allerede betalt, og enhver bivirkning (sendt email, skrevet database række, postet kommentar) er allerede sket.

Et **for-handlings** flow indsætter forhandlingen, før agenten udfører det risikable trin. Agenten foreslår handlingen, forhandlingen beslutter om den skal udføres, og kun ved godkendelse sker bivirkningen.

| Aspekt | Godkendelse efter handling (README-udsnit) | Forhandling før handling (dette notesbog) |
|---|---|---|
| Hvornår udføres godkendelsen? | Efter agenten har produceret output | Før nogen bivirkning udføres |
| LLM-omkostning ved afvisning | Allerede betalt | Betales kun for forslaget, ikke handlingen |
| Uigenkaldelige bivirkninger | Muligt (handlingen er allerede udført) | Forhindret |
| Revisionsklarhed | Godkendelsen er en print-udtalelse | Godkendelsen er en JSON-post med tidsstempel, handling, grund |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Mønster 2: Risikoniveauer

Ikke alle handlinger behøver menneskelig godkendelse. Et readonly opslag mod en offentlig API har andre risici end at sende en kundemail. At behandle begge ens spilder operatørens opmærksomhed og sænker agenten.

En simpel 3-niveau model:

| Niveau | Eksempler | Godkendelsesflow |
|---|---|---|
| `lavt` (readonly) | Søge i en vidensbase, slå flymuligheder op, hente en offentlig webside | Auto-eksekvering, logget til revision |
| `medium` (billig mutation) | Cache et resultat, øg en tæller, planlæg en påmindelse | Auto-eksekvering, men samlet til daglig gennemgang |
| `højt` (eksternt eksponeret eller irreversibel) | Send en email, opkræv et kort, post til en offentlig kanal | Bloker for menneskelig godkendelse |

Dette er et niveauinddelt system. Produktionssystemer bruger ofte mere granulære niveauer (f.eks. AWS IAM tilladelsesniveauer, rollebaserede adgangsniveauer). Den 3-niveau version nedenfor er den mindste brugbare version til en agent, der blander readonly og sideeffekthåndtering.

Klassifikatoren nedenfor bruger nøgleord heuristik, så demoen forbliver deterministisk og billig. I et produktionssystem ville du udskifte dette med en lærende klassifikator eller en politikmotor.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Mønster 3: Revisionslog og revisionssløjfe

En `print("Response approved.")` er ikke en revisionslog. For tillid bør hver portbeslutning registreres som en struktureret begivenhed, som du senere kan forespørge, afspille eller knytte til en hændelsesgennemgang.

To dele:

1. **Append-only JSONL.** Én linje per beslutning, med tidsstempel, handling, niveau, beslutning, grund. Let at søge i, let at sende til en rigtig logbutik senere.
2. **Revisionssløjfe ved afvisning.** Når porten returnerer `deny`, spørger agenten sig selv igen med afvisningsårsagen i konteksten, så det næste forslag kan undgå problemet.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Yderligere ressourcer

Flere andre offentlige projekter implementerer variationer af disse HITL-mønstre. Sammenlign tilgange for at finde ud af, hvad der passer til din stack:

- **LangChain** human-in-the-loop værktøjsindpakninger ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): drop-in værktøjsindpakninger, der standser udførelsen for menneskelig input.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ omstrukturerede dette): bruger en agentrolle specifikt til at repræsentere mennesket i multi-agent samtaler.
- **Microsoft Agent Framework (MAF)** function-invocation middleware ([docs](https://learn.microsoft.com/agent-framework/)): middleware, der kører omkring hvert værktøjs-/funktionskald, egnet til gate-logic og godkendelsesflow.

Hvert projekt håndterer de tre delmønstre forskelligt: LangChain pakker dem som værktøjer, AutoGen bruger en agentrolle, og Microsoft Agent Framework bruger function-invocation middleware. Læs en eller to implementeringer fra ende til anden, før du vælger et design til din egen agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
